In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv("input_traincopy.csv")

print("Shape:", df.shape)
df.head()

Shape: (787, 339)


,PTID_Key,EXAMDATE,T,AGE,PTGENDER,PTEDUCAT,PTETHCAT,PTRACCAT,PTMARRY,APOE4,...,ST97TS_UCSFFSX_11_02_15_UCSFFSX51_08_01_16,ST98CV_UCSFFSX_11_02_15_UCSFFSX51_08_01_16,ST98SA_UCSFFSX_11_02_15_UCSFFSX51_08_01_16,ST98TA_UCSFFSX_11_02_15_UCSFFSX51_08_01_16,ST98TS_UCSFFSX_11_02_15_UCSFFSX51_08_01_16,ST99CV_UCSFFSX_11_02_15_UCSFFSX51_08_01_16,ST99SA_UCSFFSX_11_02_15_UCSFFSX51_08_01_16,ST99TA_UCSFFSX_11_02_15_UCSFFSX51_08_01_16,ST99TS_UCSFFSX_11_02_15_UCSFFSX51_08_01_16,ST9SV_UCSFFSX_11_02_15_UCSFFSX51_08_01_16
0,8,11/1/12,11,71.4,1,18,1,1,2,0,...,0.674000,3875.000000,1528.000000,2.370000,0.562000,8909.00000,2714.000000,2.80300,0.596000,1267.000000
1,18,8/6/12,6,73.0,2,17,1,2,1,0,...,0.512144,5356.795456,1959.950778,2.280803,0.898805,13162.66608,3998.553339,2.71624,0.680506,2112.519617
2,21,12/20/12,4,55.0,2,16,1,2,1,2,...,0.532000,5047.000000,1908.000000,2.341000,0.931000,15998.00000,4388.000000,2.88100,0.910000,1838.000000
3,21,12/20/12,10,55.0,2,16,1,2,1,2,...,0.532000,5047.000000,1908.000000,2.341000,0.931000,15998.00000,4388.000000,2.88100,0.910000,1838.000000
4,21,12/20/12,22,55.0,2,16,1,2,1,2,...,0.532000,5047.000000,1908.000000,2.341000,0.931000,15998.00000,4388.000000,2.88100,0.910000,1838.000000


In [3]:
core_features = [
    'AGE', 'APOE4',
    'CDRSB', 'ADAS11', 'ADAS13',
    'RAVLT_immediate', 'RAVLT_learning',
    'RAVLT_forgetting', 'RAVLT_perc_forgetting',
    'FAQ',
    'Ventricles', 'WholeBrain', 'VN', 'ICV',
    'MMSE'
]

categorical_features = [
    'PTGENDER', 'PTEDUCAT',
    'PTETHCAT', 'PTRACCAT', 'PTMARRY'
]

extra_features = ['EXAMDATE', 'T']

selected_columns = core_features + categorical_features + extra_features

df = df[selected_columns]

print("Selected columns:", len(df.columns))

Selected columns: 22


In [4]:
df['EXAMDATE'] = pd.to_datetime(df['EXAMDATE'], errors='coerce')

df['Year'] = df['EXAMDATE'].dt.year
df['Month'] = df['EXAMDATE'].dt.month

df = df.drop(columns=['EXAMDATE'])

C:\Users\nithy\AppData\Local\Temp\ipykernel_33412\2508744399.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['EXAMDATE'] = pd.to_datetime(df['EXAMDATE'], errors='coerce')


In [5]:
df = pd.get_dummies(
    df,
    columns=['PTGENDER', 'PTETHCAT', 'PTRACCAT', 'PTMARRY'],
    drop_first=True
)

In [6]:
imputer = SimpleImputer(strategy='median')

df[df.columns] = imputer.fit_transform(df)

print("Missing values:", df.isnull().sum().sum())

Missing values: 0


In [7]:
X = df.drop(columns=['MMSE'])
y = df['MMSE']

print(X.shape)

(787, 27)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [10]:
mae = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2 = r2_score(y_test, y_pred_rf)

print("Random Forest Results:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

Random Forest Results:
MAE: 0.8732335634133604
RMSE: 1.2686987277605677
R2 Score: 0.7254392130612306


In [11]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

In [12]:
mae = mean_absolute_error(y_test, y_pred_xgb)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2 = r2_score(y_test, y_pred_xgb)

print("XGBoost Results:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

XGBoost Results:
MAE: 0.5847227073780842
RMSE: 1.1389957461501576
R2 Score: 0.7787080038024042


In [13]:
df.columns.tolist()

['AGE',
 'APOE4',
 'CDRSB',
 'ADAS11',
 'ADAS13',
 'RAVLT_immediate',
 'RAVLT_learning',
 'RAVLT_forgetting',
 'RAVLT_perc_forgetting',
 'FAQ',
 'Ventricles',
 'WholeBrain',
 'VN',
 'ICV',
 'MMSE',
 'PTEDUCAT',
 'T',
 'Year',
 'Month',
 'PTGENDER_2',
 'PTETHCAT_1',
 'PTETHCAT_2',
 'PTRACCAT_2',
 'PTRACCAT_3',
 'PTRACCAT_4',
 'PTMARRY_2',
 'PTMARRY_3',
 'PTMARRY_4']

In [14]:
# Extract min-max for required columns

required_cols = [
    'RAVLT_immediate', 'RAVLT_learning',
    'RAVLT_forgetting', 'RAVLT_perc_forgetting',
    'ADAS11', 'ADAS13',
    'CDRSB', 'FAQ'
]

df_stats = {}

for col in required_cols:
    df_stats[col] = {
        'min': df[col].min(),
        'max': df[col].max()
    }

print(df_stats)

{'RAVLT_immediate': {'min': 6.0, 'max': 73.0}, 'RAVLT_learning': {'min': -3.0, 'max': 12.0}, 'RAVLT_forgetting': {'min': -3.839171664, 'max': 15.0}, 'RAVLT_perc_forgetting': {'min': -129.4301515, 'max': 166.1443048}, 'ADAS11': {'min': -0.31622351, 'max': 40.0}, 'ADAS13': {'min': -1.542427884, 'max': 51.0}, 'CDRSB': {'min': -0.140128465, 'max': 11.0}, 'FAQ': {'min': -0.264116841, 'max': 29.67784631}}


In [15]:
def normalize(value, min_val, max_val):
    if max_val - min_val == 0:
        return 0
    return (value - min_val) / (max_val - min_val)

In [16]:
def scale_to_clinical(norm_value, col_name, df_stats):
    col_min = df_stats[col_name]['min']
    col_max = df_stats[col_name]['max']
    
    return col_min + norm_value * (col_max - col_min)

In [17]:
def game_to_clinical(game_data):
    
    acc = game_data['accuracy'] / 100
    mistakes = game_data['mistakes'] / game_data['max_mistakes']
    time = game_data['time_taken'] / game_data['max_time']
    learning = game_data['learning_score'] / game_data['max_learning']
    
    clinical = {}
    
    # -------------------------
    # MEMORY (direct mapping)
    # -------------------------
    clinical['RAVLT_immediate'] = acc * 60
    
    clinical['RAVLT_learning'] = learning * 10
    
    clinical['RAVLT_forgetting'] = mistakes * 15
    
    clinical['RAVLT_perc_forgetting'] = mistakes * 100
    
    
    # -------------------------
    # COGNITIVE (independent signals)
    # -------------------------
    clinical['ADAS11'] = (
        0.5 * mistakes +
        0.3 * time +
        0.2 * (1 - learning)
    ) * 30
    
    clinical['ADAS13'] = (
        0.5 * mistakes +
        0.3 * time +
        0.2 * (1 - learning)
    ) * 60
    
    
    # -------------------------
    # FUNCTIONAL
    # -------------------------
    clinical['CDRSB'] = time * 18
    
    clinical['FAQ'] = (mistakes + time) / 2 * 30
    
    return clinical

In [18]:
def add_static_features(clinical_data, df):
    
    return {
        'AGE': 65,
        'APOE4': 0,
        
        'Ventricles': df['Ventricles'].mean(),
        'WholeBrain': df['WholeBrain'].mean(),
        'VN': df['VN'].mean(),
        'ICV': df['ICV'].mean(),
        
        'T': 0
    } | clinical_data

In [19]:
def prepare_model_input(game_data, df, df_stats, X_columns):
    
    # Step 1: map game → clinical
    clinical = game_to_clinical(game_data)
    
    # Step 2: add static features
    full_input = add_static_features(clinical, df)
    
    # Step 3: convert to dataframe
    import pandas as pd
    input_df = pd.DataFrame([full_input])
    
    # Step 4: align with training columns
    input_df = input_df.reindex(columns=X_columns, fill_value=0)
    
    return input_df

In [20]:
game_data = {
    'accuracy': 35,
    'mistakes': 8,
    'max_mistakes': 10,
    'time_taken': 55,
    'max_time': 60,
    'learning_score': 2,
    'max_learning': 10
}

In [21]:
input_df = prepare_model_input(game_data, df, df_stats, X.columns)

predicted_mmse = xgb.predict(input_df)[0]

print("Predicted MMSE:", predicted_mmse)

Predicted MMSE: 22.181395


In [22]:
def mmse_to_severity(mmse):
    if mmse >= 24:
        return "Normal"
    elif mmse >= 18:
        return "Mild"
    else:
        return "Severe"

print("Severity:", mmse_to_severity(predicted_mmse))

Severity: Mild


In [23]:
print(X.describe())

              AGE       APOE4       CDRSB      ADAS11      ADAS13  \
count  787.000000  787.000000  787.000000  787.000000  787.000000   
mean    71.804956    0.534943    1.366636    9.005778   13.996912   
std      7.014455    0.674687    1.695110    6.368455    9.323943   
min     55.000000    0.000000   -0.140128   -0.316224   -1.542428   
25%     67.250000    0.000000    0.000000    4.943694    7.000000   
50%     71.500000    0.000000    0.500007    7.000000   12.000000   
75%     76.400000    1.000000    2.000012   11.000156   18.000000   
max     90.300000    2.000000   11.000000   40.000000   51.000000   

       RAVLT_immediate  RAVLT_learning  RAVLT_forgetting  \
count       787.000000      787.000000        787.000000   
mean         38.821938        4.619793          4.325218   
std          13.260615        2.753345          2.664850   
min           6.000000       -3.000000         -3.839172   
25%          29.000000        3.000000          3.000000   
50%          38.00

In [24]:
print(input_df.T)

                                  0
AGE                    6.500000e+01
APOE4                  0.000000e+00
CDRSB                  1.650000e+01
ADAS11                 2.505000e+01
ADAS13                 5.010000e+01
RAVLT_immediate        2.100000e+01
RAVLT_learning         2.000000e+00
RAVLT_forgetting       1.200000e+01
RAVLT_perc_forgetting  8.000000e+01
FAQ                    2.575000e+01
Ventricles             3.497565e+04
WholeBrain             1.031790e+06
VN                     2.305995e-02
ICV                    1.486208e+06
PTEDUCAT               0.000000e+00
T                      0.000000e+00
Year                   0.000000e+00
Month                  0.000000e+00
PTGENDER_2             0.000000e+00
PTETHCAT_1             0.000000e+00
PTETHCAT_2             0.000000e+00
PTRACCAT_2             0.000000e+00
PTRACCAT_3             0.000000e+00
PTRACCAT_4             0.000000e+00
PTMARRY_2              0.000000e+00
PTMARRY_3              0.000000e+00
PTMARRY_4              0.000

In [25]:
print(X.describe())

              AGE       APOE4       CDRSB      ADAS11      ADAS13  \
count  787.000000  787.000000  787.000000  787.000000  787.000000   
mean    71.804956    0.534943    1.366636    9.005778   13.996912   
std      7.014455    0.674687    1.695110    6.368455    9.323943   
min     55.000000    0.000000   -0.140128   -0.316224   -1.542428   
25%     67.250000    0.000000    0.000000    4.943694    7.000000   
50%     71.500000    0.000000    0.500007    7.000000   12.000000   
75%     76.400000    1.000000    2.000012   11.000156   18.000000   
max     90.300000    2.000000   11.000000   40.000000   51.000000   

       RAVLT_immediate  RAVLT_learning  RAVLT_forgetting  \
count       787.000000      787.000000        787.000000   
mean         38.821938        4.619793          4.325218   
std          13.260615        2.753345          2.664850   
min           6.000000       -3.000000         -3.839172   
25%          29.000000        3.000000          3.000000   
50%          38.00